# Task 1

Fit a linear regression which describes tree age (Dataset dab.csv). Use numpy.linalg library to solve the equation $X^TX=X^Ty$. Present dataset and the model on a picture.

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [ ]:
# loading data and dropping missing values
Data = pd.read_csv('dab.csv')
Data = pd.DataFrame(Data).dropna()
# changing second data type from string to float
Data['srednica2'] = Data['srednica2'].str.replace(',', '.').astype(float) # vectorized operation



In [ ]:
# checking linearity
ratios = Data['srednica'] / Data['srednica2']
np.ptp(ratios) # return max - min (peak to peak) value 
# we see that difference between ratios of this value columns are really small, we can assume they are colinear

In [ ]:
# changing data type 
np_srednica = Data['srednica'].to_numpy().reshape(-1, 1)
# creating matrix with beta intercept coefficient
X = np.hstack([np.ones((np_srednica.shape[0], 1)), np_srednica])
b = Data['wiek'].to_numpy().reshape(-1, 1)
# solving equation
intercept_solve, coef_solve = np.linalg.solve(X.T @ X, X.T @ b)

In [ ]:
x_plot = np_srednica.reshape(-1, )
y_plot = b.reshape(-1, )

plt.plot(x_plot, coef_solve * x_plot + intercept_solve)
plt.scatter(x_plot, y_plot)
plt.title(f'manual coefficent = {coef_solve}')
RSS = sum((x_r - x_e)**2 for x_r, x_e in zip())

# Task 2

Perform the above task using the function $\texttt{LinearRegression()}$ from $\verb+sklearn.linear_model+$.

In [ ]:
# uwaga to bierze za argument df !!! a potem 
X_train, X_test, y_train, y_test = train_test_split(Data[['srednica']], Data['wiek'], test_size=0.2)
model = LinearRegression()
model.fit(X_train, y_train)
model.score(X_test, y_test)

In [ ]:
# comparison 
model = LinearRegression()
model.fit(Data[['srednica']], Data['wiek'])
model.coef_, coef_solve, model.intercept_, intercept_solve

# Task 3

Create an artificial dataset ($x_i,y_i$, with $n=1000$) such that $y=x^2+e$, where $e$ comes from the standard normal distribution and $x\in[-1,1]$. 
Next, split the dataset into two parts (for $x<0$ and $x>0$) and train a linear regression $y=\beta_0 + \beta_1x$ on the first part. Please, read coefficients, $R^2$ and draw histogram of residuals.

Calculate the mean squared error (MSE) $\frac{\sum_{i=1}^{n/2}(\hat{y_i}-y_i)^2}{n/2}$ for the values fitted on the traning set($x<0$) and (next) on the test set($x>0$). Please compare this two values.

In [ ]:
# parameters 
n = 1000
seed = 3
scale_par = 1
# creating data set
x = stats.uniform(loc=-1, scale= 2).rvs(n, random_state=seed).reshape(-1, 1)
y = x + stats.norm(scale=scale_par).rvs(n, random_state=seed).reshape(-1, 1)
df = pd.DataFrame(data=np.hstack((x, y)), columns=['X', 'Y'])

# splitting sets 
filt = df['X'] > 0
df_pos = df[filt]
# plot 
plt.scatter(x, y)

# to recap - we need 1 dim data for filters !!!
# random state - lower random state give ous worse results (on R^2 ~ significant ) 


In [ ]:
# learning model on data X > 0 
model = LinearRegression()
model.fit(df_pos[['X']], df_pos['Y'])
# getting scores (on all data)
score = model.score(df[['X']], df['Y'])  
coef, intercept = model.coef_.item(), model.intercept_
# residuals differences between true value, estimated value
df['Residuals'] = df[['Y']] - model.predict(df[['X']]).reshape(-1, 1)
# printing values
print(f'Score: {score}\nCoefficent: {coef}\nIntercept: {intercept}')
plt.hist(df[['Residuals']], bins=20, edgecolor='black')

# TODO ~ does we can estimate error based on residual value

In [ ]:
# calculating MSE 


# for x > 0, to know sklearn.metrics have this functions implemented
MSE_pos = (df['Residuals'][filt]**2).sum() / len(filt)
MSE_neg = (df['Residuals'][~filt]**2).sum() / len(~filt)
print(f'positive: {MSE_pos}\nnegative: {MSE_neg}')

# Task 4

Repeat Task 3 for some better linear regression model (you can use a data transformation $\texttt{PolynomialFeatures}$). 

Again read $R^2$ and MSE for fitted values on both parts of data.



# Task 5

In [ ]:
from sklearn.model_selection import cross_val_score

Perform cross-validation  $\texttt{cross_val_score}$ for models from Task 1,2

In [ ]:
model = LinearRegression()
scores = cross_val_score(model, Data[['srednica']], Data['wiek'], cv=5)
scores